## gamelog_cleaning¶
- prepping and cleaning Lebron's gamelog csv stats in prep for further SQL querying

Remember:

Pandas is Data Cleaning
Matplotlib is Data visuals
Numpy is Math Operations/Calculator

In [2]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
import psycopg2
from pandasql import sqldf
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
pd.options.plotting.backend = 'plotly'
from bokeh.plotting import figure, show
from bokeh.io import output_notebook
import voila
import warnings
warnings.filterwarnings('ignore')

In [7]:
#importing the csv data into python
#store it as a dataframe df
# Reading the path piece by piece:
#.. goes up one folder, from notebooks/ to lebron-data-story/
#/data goes into the data folder
#/raw goes into the raw folder
#/lebron_per_game.csv is the file

df = pd.read_csv ('../data/raw/lebron_gamelog_2010_11.csv')
df

,Rk,Gcar,Gtm,Date,Team,Unnamed: 5,Opp,Result,GS,MP,...,DRB,TRB,AST,STL,BLK,TOV,PF,PTS,GmSc,+/-
0,1.0,549.0,1.0,10/26/10,MIA,@,BOS,L 80-88,*,42:39,...,4,4,3,1,2,8,1,31,16,1
1,2.0,550.0,2.0,10/27/10,MIA,@,PHI,W 97-87,*,37:33,...,6,6,7,2,1,9,1,16,9.6,6
2,3.0,551.0,3.0,10/29/10,MIA,NaN,ORL,W 96-70,*,30:02,...,6,6,7,0,0,3,4,15,10,13
3,4.0,552.0,4.0,10/31/10,MIA,@,NJN,W 101-78,*,28:57,...,7,7,7,2,0,5,3,20,15.6,18
4,5.0,553.0,5.0,11/2/10,MIA,NaN,MIN,W 129-97,*,32:26,...,0,0,12,2,1,1,0,20,23.7,32
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
78,77.0,625.0,79.0,4/8/11,MIA,NaN,CHA,W 112-103,*,37:23,...,6,7,9,2,1,1,2,23,24.7,12
79,78.0,626.0,80.0,4/10/11,MIA,NaN,BOS,W 100-77,*,37:11,...,5,5,7,4,1,4,1,27,23.2,24
80,79.0,627.0,81.0,4/11/11,MIA,@,ATL,W 98-90,*,42:39,...,9,10,7,3,0,7,1,34,26.4,7
81,79.0,NaN,82.0,4/13/11,MIA,@,TOR,W 97-79,Did Not Play,Did Not Play,...,Did Not Play,Did Not Play,Did Not Play,Did Not Play,Did Not Play,Did Not Play,Did Not Play,Did Not Play,Did Not Play,Did Not Play


In [8]:
dfcopy = df.copy()
dfcopy

,Rk,Gcar,Gtm,Date,Team,Unnamed: 5,Opp,Result,GS,MP,...,DRB,TRB,AST,STL,BLK,TOV,PF,PTS,GmSc,+/-
0,1.0,549.0,1.0,10/26/10,MIA,@,BOS,L 80-88,*,42:39,...,4,4,3,1,2,8,1,31,16,1
1,2.0,550.0,2.0,10/27/10,MIA,@,PHI,W 97-87,*,37:33,...,6,6,7,2,1,9,1,16,9.6,6
2,3.0,551.0,3.0,10/29/10,MIA,NaN,ORL,W 96-70,*,30:02,...,6,6,7,0,0,3,4,15,10,13
3,4.0,552.0,4.0,10/31/10,MIA,@,NJN,W 101-78,*,28:57,...,7,7,7,2,0,5,3,20,15.6,18
4,5.0,553.0,5.0,11/2/10,MIA,NaN,MIN,W 129-97,*,32:26,...,0,0,12,2,1,1,0,20,23.7,32
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
78,77.0,625.0,79.0,4/8/11,MIA,NaN,CHA,W 112-103,*,37:23,...,6,7,9,2,1,1,2,23,24.7,12
79,78.0,626.0,80.0,4/10/11,MIA,NaN,BOS,W 100-77,*,37:11,...,5,5,7,4,1,4,1,27,23.2,24
80,79.0,627.0,81.0,4/11/11,MIA,@,ATL,W 98-90,*,42:39,...,9,10,7,3,0,7,1,34,26.4,7
81,79.0,NaN,82.0,4/13/11,MIA,@,TOR,W 97-79,Did Not Play,Did Not Play,...,Did Not Play,Did Not Play,Did Not Play,Did Not Play,Did Not Play,Did Not Play,Did Not Play,Did Not Play,Did Not Play,Did Not Play


In [13]:
dfcopy.shape

#83 rows and 34 columns

(83, 34)

In [15]:
dfcopy.head(10)

#.head(10) gives me the first 10 rows of the dataset

,Rk,Gcar,Gtm,Date,Team,Unnamed: 5,Opp,Result,GS,MP,...,DRB,TRB,AST,STL,BLK,TOV,PF,PTS,GmSc,+/-
0,1.0,549.0,1.0,10/26/10,MIA,@,BOS,L 80-88,*,42:39,...,4,4,3,1,2,8,1,31,16,1
1,2.0,550.0,2.0,10/27/10,MIA,@,PHI,W 97-87,*,37:33,...,6,6,7,2,1,9,1,16,9.6,6
2,3.0,551.0,3.0,10/29/10,MIA,NaN,ORL,W 96-70,*,30:02,...,6,6,7,0,0,3,4,15,10,13
3,4.0,552.0,4.0,10/31/10,MIA,@,NJN,W 101-78,*,28:57,...,7,7,7,2,0,5,3,20,15.6,18
4,5.0,553.0,5.0,11/2/10,MIA,NaN,MIN,W 129-97,*,32:26,...,0,0,12,2,1,1,0,20,23.7,32
5,6.0,554.0,6.0,11/5/10,MIA,@,NOH,L 93-96,*,41:12,...,7,7,10,3,1,3,0,20,20.6,-6
6,7.0,555.0,7.0,11/6/10,MIA,NaN,NJN,W 101-89,*,33:35,...,3,3,9,2,0,4,1,23,19.4,11
7,8.0,556.0,8.0,11/9/10,MIA,NaN,UTA,L 114-116 (OT),*,46:44,...,9,11,14,1,3,2,3,20,22.8,4
8,9.0,557.0,9.0,11/11/10,MIA,NaN,BOS,L 107-112,*,44:28,...,10,10,9,3,0,4,3,35,29,0
9,10.0,558.0,10.0,11/13/10,MIA,NaN,TOR,W 109-100,*,39:20,...,1,2,11,2,0,3,0,23,17.6,11


In [21]:
dfcopy.tail()

#.tail(10) gives me the last 10 rows of the dataset

,Rk,Gcar,Gtm,Date,Team,Unnamed: 5,Opp,Result,GS,MP,...,DRB,TRB,AST,STL,BLK,TOV,PF,PTS,GmSc,+/-
78,77.0,625.0,79.0,4/8/11,MIA,NaN,CHA,W 112-103,*,37:23,...,6,7,9,2,1,1,2,23,24.7,12
79,78.0,626.0,80.0,4/10/11,MIA,NaN,BOS,W 100-77,*,37:11,...,5,5,7,4,1,4,1,27,23.2,24
80,79.0,627.0,81.0,4/11/11,MIA,@,ATL,W 98-90,*,42:39,...,9,10,7,3,0,7,1,34,26.4,7
81,79.0,NaN,82.0,4/13/11,MIA,@,TOR,W 97-79,Did Not Play,Did Not Play,...,Did Not Play,Did Not Play,Did Not Play,Did Not Play,Did Not Play,Did Not Play,Did Not Play,Did Not Play,Did Not Play,Did Not Play
82,NaN,NaN,NaN,NaN,NaN,NaN,NaN,57-22,79,3064,...,510,590,554,124,50,284,163,2111,21.7,621


In [17]:
dfcopy.info()

<class 'pandas.DataFrame'>
RangeIndex: 83 entries, 0 to 82
Data columns (total 34 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Rk          82 non-null     float64
 1   Gcar        79 non-null     float64
 2   Gtm         82 non-null     float64
 3   Date        82 non-null     str    
 4   Team        82 non-null     str    
 5   Unnamed: 5  41 non-null     str    
 6   Opp         82 non-null     str    
 7   Result      83 non-null     str    
 8   GS          83 non-null     str    
 9   MP          83 non-null     str    
 10  FG          83 non-null     str    
 11  FGA         83 non-null     str    
 12  FG%         83 non-null     str    
 13  3P          83 non-null     str    
 14  3PA         83 non-null     str    
 15  3P%         81 non-null     str    
 16  2P          83 non-null     str    
 17  2PA         83 non-null     str    
 18  2P%         83 non-null     str    
 19  eFG%        83 non-null     str    
 20  F

In [18]:
dfcopy.dtypes

#.dtypes gives me the datatypes per each column

Rk            float64
Gcar          float64
Gtm           float64
Date              str
Team              str
Unnamed: 5        str
Opp               str
Result            str
GS                str
MP                str
FG                str
FGA               str
FG%               str
3P                str
3PA               str
3P%               str
2P                str
2PA               str
2P%               str
eFG%              str
FT                str
FTA               str
FT%               str
ORB               str
DRB               str
TRB               str
AST               str
STL               str
BLK               str
TOV               str
PF                str
PTS               str
GmSc              str
+/-               str
dtype: object

In [20]:
dfcopy.isnull().sum()

#.isnull().sum() gives me any missing values within the dataset

Rk             1
Gcar           4
Gtm            1
Date           1
Team           1
Unnamed: 5    42
Opp            1
Result         0
GS             0
MP             0
FG             0
FGA            0
FG%            0
3P             0
3PA            0
3P%            2
2P             0
2PA            0
2P%            0
eFG%           0
FT             0
FTA            0
FT%            0
ORB            0
DRB            0
TRB            0
AST            0
STL            0
BLK            0
TOV            0
PF             0
PTS            0
GmSc           0
+/-            0
dtype: int64

### Observations from data exploration
- this gamelog dataset shows LeBron's stat for every single game of the 2010-2011 NBA season
- the last row, 83rd row, gives Lebron's totals for this specific season.
- 2010-2011 was Lebron's first season with the Miami Heat

- in this 2010-2011 gamelog dataset, there are 83 rows with 34 columns
- from the .tail() the Heat played 82 games with a season record of 57-22 in the games Lebron played.
- Using the MP column, I can see that Lebron missed 3 games this season. Game 41,42, and 82 he DNP

Must do after data exploration:
- remove the totals row
- keep the games he did not play. So, I will create a status column having values like 'played' and 'did not play'
- edit and rename unnamed:5 to home_away, with valuesl like home/away
- the MP (minutes played) column is a string data type. Which will become problematic for averages and more since it is not in number format. Will need to find a way to switch this data type from string to integer. which is why there are 82 columns because there are 82 games in a season. convert to decimal minutes
- date text is under ex. 10/26/10 need to convert to sql fromat yyyy/mm/dd. Convert to real date.
- i want to create a win_loss column. Win or Loss column as a RESULT column
- convert the stat columns to numbers since it is currently a string
- rename columns for sql. lowercase with no digits. ex. 3p% turn into three_p
- add a season column

In [22]:
# removing the totals row . the row with no team game number
dfcopy = dfcopy[dfcopy['Gtm'].notna()].copy()

In [ ]:
#dfcopy['Gtm'] picks the team game number column
#.notna() means not missing. True if Gtm has a value, and false if empty. 82 trues and 1 false
#dfcopy[..] means that it will keep only the true rows
#.copy() makes the results in a new dataframe

In [23]:
dfcopy.shape
# it is now 82 rows instead of 83

(82, 34)

### Data cleaning function:
  - After data exploration and figuring out what i want to clean and need to add and get rid of, i move on to the data cleaning portion. 
   - This is where I will utilize AI/Claude to help me

In [26]:
df

,Rk,Gcar,Gtm,Date,Team,Unnamed: 5,Opp,Result,GS,MP,...,DRB,TRB,AST,STL,BLK,TOV,PF,PTS,GmSc,+/-
0,1.0,549.0,1.0,10/26/10,MIA,@,BOS,L 80-88,*,42:39,...,4,4,3,1,2,8,1,31,16,1
1,2.0,550.0,2.0,10/27/10,MIA,@,PHI,W 97-87,*,37:33,...,6,6,7,2,1,9,1,16,9.6,6
2,3.0,551.0,3.0,10/29/10,MIA,NaN,ORL,W 96-70,*,30:02,...,6,6,7,0,0,3,4,15,10,13
3,4.0,552.0,4.0,10/31/10,MIA,@,NJN,W 101-78,*,28:57,...,7,7,7,2,0,5,3,20,15.6,18
4,5.0,553.0,5.0,11/2/10,MIA,NaN,MIN,W 129-97,*,32:26,...,0,0,12,2,1,1,0,20,23.7,32
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
78,77.0,625.0,79.0,4/8/11,MIA,NaN,CHA,W 112-103,*,37:23,...,6,7,9,2,1,1,2,23,24.7,12
79,78.0,626.0,80.0,4/10/11,MIA,NaN,BOS,W 100-77,*,37:11,...,5,5,7,4,1,4,1,27,23.2,24
80,79.0,627.0,81.0,4/11/11,MIA,@,ATL,W 98-90,*,42:39,...,9,10,7,3,0,7,1,34,26.4,7
81,79.0,NaN,82.0,4/13/11,MIA,@,TOR,W 97-79,Did Not Play,Did Not Play,...,Did Not Play,Did Not Play,Did Not Play,Did Not Play,Did Not Play,Did Not Play,Did Not Play,Did Not Play,Did Not Play,Did Not Play


In [35]:
def clean_gamelog(file_path,season):
    df = pd.read_csv(file_path)

    # 1. remove the totals row (the only row with no team game number)
    df = df[df['Gtm'].notna()].copy()

    # 2. status: 'Played', or the reason he didn't play (Gcar is blank when he didn't play)
    # Gcar stands for career game number
    df['status'] = np.where(df['Gcar'].isna(), df['GS'], 'Played')

    # started: Yes if GS is '*', otherwise No
    #GS stands for games started. Lebron has always mostly started since he started his career
    df['started'] = np.where(df['GS'] == '*', 'Yes', 'No')
    df = df.drop(columns=['GS'])

    # 3. home_away: '@' means away, blank means home
    df = df.rename(columns={'Unnamed: 5': 'home_away'})
    df['home_away'] = np.where(df['home_away'] == '@', 'Away', 'Home')

    # 4. split Result ('L 80-88') into win_loss ('L') and score ('80-88')
    df['win_loss'] = df['Result'].str[0]
    df['score'] = df['Result'].str[2:]
    df = df.drop(columns=['Result'])

    # 5. MP: '42:39' -> 42.65 decimal minutes (missed games become empty)
    parts = df['MP'].str.split(':', expand=True)
    minutes = pd.to_numeric(parts[0], errors='coerce')
    seconds = pd.to_numeric(parts[1], errors='coerce')
    df['MP'] = (minutes + seconds / 60).round(2)

    # 6. Date: '10/26/10' -> a real date (2010-10-26)
    df['Date'] = pd.to_datetime(df['Date'], format='%m/%d/%y')

    # 7. stat columns to numbers ('Did Not Play' text becomes empty)
    text_cols = ['Date', 'Team', 'home_away', 'Opp', 'win_loss', 'score', 'status', 'started']
    num_cols = [col for col in df.columns if col not in text_cols]
    df[num_cols] = df[num_cols].apply(pd.to_numeric, errors='coerce')

    # 8. SQL-friendly column names
    df.columns = (
        df.columns
        .str.lower()
        .str.replace('%', '_pct')
        .str.replace('3p', 'three_p')
        .str.replace('2p', 'two_p')
        .str.replace('+/-', 'plus_minus')
    )

    # 9. add the season
    df['season'] = season

    return df

In [ ]:
# now this function i will use for each of the three game logs

In [36]:
g1 = clean_gamelog('../data/raw/lebron_gamelog_2010_11.csv', '2010-11')
g2 = clean_gamelog('../data/raw/lebron_gamelog_2011_12.csv', '2011-12')
g3 = clean_gamelog('../data/raw/lebron_gamelog_2012_13.csv', '2012-13')

In [38]:
#stack the three game logs into one table

In [37]:
games = pd.concat([g1, g2, g3], ignore_index=True)
games.shape
#230 rows and 37 columns

(230, 37)

In [ ]:
# data checking the work

In [40]:
games['status'].value_counts()

# results: 217 Played, 7 Did Not Dress, 4 Did Not Play, 2 Inactive.

status
Played           217
Did Not Dress      7
Did Not Play       4
Inactive           2
Name: count, dtype: int64

In [41]:
games.groupby('season')['win_loss'].value_counts()

# results: 2010-11: 58 W / 24 L , 2011-12: 46 W / 20 L , 2012-13: 66 W / 16 L

season   win_loss
2010-11  W           58
         L           24
2011-12  W           46
         L           20
2012-13  W           66
         L           16
Name: count, dtype: int64

In [42]:
games.groupby('season')['pts'].mean().round(1)

# results: 26.7 , 27.1 , 26.8

season
2010-11    26.7
2011-12    27.1
2012-13    26.8
Name: pts, dtype: float64

In [43]:
# saving the 'games' file to csv for sql as lebron_gamelogs.csv
games.to_csv('../data/processed/lebron_gamelogs.csv', index=False)

In [45]:
games.info()

<class 'pandas.DataFrame'>
RangeIndex: 230 entries, 0 to 229
Data columns (total 37 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   rk           230 non-null    float64       
 1   gcar         217 non-null    float64       
 2   gtm          230 non-null    float64       
 3   date         230 non-null    datetime64[us]
 4   team         230 non-null    str           
 5   home_away    230 non-null    str           
 6   opp          230 non-null    str           
 7   mp           217 non-null    float64       
 8   fg           217 non-null    float64       
 9   fga          217 non-null    float64       
 10  fg_pct       217 non-null    float64       
 11  three_p      217 non-null    float64       
 12  three_pa     217 non-null    float64       
 13  three_p_pct  200 non-null    float64       
 14  two_p        217 non-null    float64       
 15  two_pa       217 non-null    float64       
 16  two_p_pct    217 no